# House Price Prediction: Model Training and Exploration

This notebook demonstrates the end-to-end process of training the Linear Regression model used in the House Price Estimator application. 

### Pipeline Stages:
1. **Load and Explore**: Load the dataset and analyze column data types and statistics.
2. **Data Cleaning**: Handle duplicates, check for missing values, and analyze outlier bounds using IQR.
3. **Exploratory Data Analysis (EDA)**: Visualize correlations, distributions, and feature relationships.
4. **Preprocessing**: Perform train-test splits and scale features using `StandardScaler`.
5. **Model Training**: Train a `LinearRegression` model from `scikit-learn`.
6. **Evaluation**: Assess accuracy using R2 score and Root Mean Squared Error (RMSE).
7. **Interpretation**: Analyze model coefficients and check for multicollinearity.
8. **Serialization**: Save the trained model, scaler, and performance metrics for the API backend.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Set plotting aesthetics
sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 120
})
PALETTE = ['#1B4332', '#B87333', '#D4A574', '#2D3436', '#5C8A4D']

## 1. Load and Explore Data

In [ ]:
# Load dataset
df = pd.read_csv('../data/housing.csv')

# Display basic details
print(f"Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

## 2. Data Cleaning & Outlier Analysis

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

# Check for duplicate rows
print(f"\nDuplicate rows found: {df.duplicated().sum()}")

In [ ]:
# Outlier detection using the Interquartile Range (IQR) method
features = ['Area Income', 'Area House Age', 'Area No of Rooms', 'Area No of Bedrooms', 'Area Population']

for col in features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: {outliers} outliers (valid range: {lower:.1f} to {upper:.1f})")

print("\nDecision: Retain outliers since they represent realistic deviations in regional metrics.")

## 3. Exploratory Data Analysis

In [ ]:
# Pairwise Correlation Heatmap
plt.figure(figsize=(8, 6))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.3f', cmap='YlOrBr', linewidths=0.5, square=True)
plt.title('Feature Correlation Heatmap', fontweight='bold', pad=15)
plt.show()

In [ ]:
# Feature relationships vs Price
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
scatter_features = ['Area Income', 'Area No of Rooms', 'Area Population']

for i, feat in enumerate(scatter_features):
    axes[i].scatter(df[feat], df['Price'], alpha=0.3, color=PALETTE[i], s=10)
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Price ($)')
    axes[i].set_title(f'{feat} vs Price', fontweight='bold')
    # Fit a simple trend line
    z = np.polyfit(df[feat], df['Price'], 1)
    p = np.poly1d(z)
    x_sorted = np.sort(df[feat])
    axes[i].plot(x_sorted, p(x_sorted), color='#B87333', linewidth=2, linestyle='--')

plt.suptitle('Key Feature Relationships with Price Trendlines', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Price Distribution
plt.figure(figsize=(10, 5))
plt.hist(df['Price'], bins=50, color=PALETTE[0], edgecolor='white', alpha=0.85)
plt.axvline(df['Price'].mean(), color=PALETTE[1], linestyle='--', linewidth=2, label=f"Mean: ${df['Price'].mean():,.0f}")
plt.axvline(df['Price'].median(), color=PALETTE[2], linestyle='--', linewidth=2, label=f"Median: ${df['Price'].median():,.0f}")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.title('Distribution of target House Prices', fontweight='bold')
plt.legend()
plt.show()

## 4. Train-Test Split and Feature Scaling

In [ ]:
X = df[features]
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set samples: {X_train.shape[0]}")
print(f"Testing set samples:  {X_test.shape[0]}")

In [ ]:
# Scale inputs using StandardScaler fitted only on the training set
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 5. Model Training

In [ ]:
# Fit Linear Regression
model = LinearRegression()
model.fit(X_train_scaled, y_train)
print("Model trained successfully.")

## 6. Evaluation

In [ ]:
y_pred = model.predict(X_test_scaled)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
print(f"R2 Score: {r2:.4f} ({r2*100:.1f}% variance explained)")

## 7. Feature Coefficient Interpretation

In [ ]:
# Fit on raw data to inspect direct coefficients per unit increase
model_unscaled = LinearRegression()
model_unscaled.fit(X_train, y_train)

print(f"Intercept: ${model_unscaled.intercept_:,.2f}\n")
coef_df = pd.DataFrame({
    'Feature': features,
    'Coefficient ($ value change)': model_unscaled.coef_
}).sort_values('Coefficient ($ value change)', ascending=False)
coef_df

In [ ]:
# Multicollinearity check between total rooms and bedrooms
corr_rooms_bed = df['Area No of Rooms'].corr(df['Area No of Bedrooms'])
print(f"Correlation between Number of Rooms and Number of Bedrooms: {corr_rooms_bed:.3f}")

## 8. Save Artifacts

In [ ]:
# Serialize for production API usage
os.makedirs('../model', exist_ok=True)

joblib.dump(model, '../model/house_price_model.pkl')
joblib.dump(scaler, '../model/scaler.pkl')
joblib.dump(features, '../model/feature_names.pkl')
joblib.dump({'rmse': rmse, 'r2': r2}, '../model/metrics.pkl')

print("All artifacts successfully saved to the model/ directory.")